# 🎯 DICE — Degradation & Image Classification Engine
### Convolutional Neural Network for Aerial Image Quality Classification

**Author:** Aniket Kumar  
**Task:** Classify drone images into three quality categories:
- `Class 0` — Clean (original, high-quality)
- `Class 1` — Blurred (simulated lens defocus)
- `Class 2` — Noisy (simulated sensor noise)

**Model:** Custom CNN trained with TensorFlow / Keras  
**Dataset:** VisDrone aerial imagery (~6,471 images × 3 classes = 19,413 total)

---

## Cell 1 — Setup & Data Loading

In [ ]:
# ==============================================================================
# CELL 1 — SETUP & DATA LOADING
# ==============================================================================
#
# Here we import every library we'll need for the entire notebook and load the
# DICE dataset from disk into TensorFlow Dataset objects.
# ==============================================================================

# ── Core scientific libraries ──────────────────────────────────────────────────
import numpy as np          # Numerical arrays (used for confusion matrix data)
import matplotlib.pyplot as plt  # Plotting training curves and the heatmap
import seaborn as sns       # High-level wrapper around matplotlib — beautiful heatmaps

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers  # Individual building blocks (Conv2D, Dense, etc.)

# ── Scikit-learn — metrics only (no training happens here) ────────────────────
from sklearn.metrics import confusion_matrix, classification_report


print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print("All libraries loaded successfully.\n")


# ==============================================================================
# CONFIGURATION — change these values in one place and everything updates below
# ==============================================================================

DATASET_DIR  = "./dataset"   # Root folder containing the three class sub-folders
IMAGE_SIZE   = (256, 256)    # Every image will be resized to this (height, width)
BATCH_SIZE   = 32            # Number of images processed per gradient-update step
NUM_CLASSES  = 3             # Class_0_Clean, Class_1_Blurred, Class_2_Noisy
RANDOM_SEED  = 42            # Reproducibility — same split every run
EPOCHS       = 10            # Full passes over the training data
MODEL_PATH   = "dice_model.keras"  # Where the trained model will be saved

# Class names in the exact alphabetical order TensorFlow will assign them
CLASS_NAMES = ["Class_0_Clean", "Class_1_Blurred", "Class_2_Noisy"]


# ==============================================================================
# LOADING THE DATASET
# ==============================================================================
#
# tf.keras.utils.image_dataset_from_directory() scans a directory whose
# sub-folders are the class labels and builds a tf.data.Dataset automatically.
#
# Key parameters:
#   directory       : Root folder with one sub-folder per class.
#   validation_split: Fraction of data held out for validation (0.2 = 20%).
#   subset          : Which portion to return ('training' or 'validation').
#   seed            : Random seed so the split is identical across runs.
#   image_size      : All images are resized to this before batching.
#   batch_size      : How many images to stack into one tensor at a time.
#   label_mode      : 'int' gives integer class indices (0, 1, 2) which is
#                     required for SparseCategoricalCrossentropy loss.
# ==============================================================================

train_ds = tf.keras.utils.image_dataset_from_directory(
    directory        = DATASET_DIR,
    validation_split = 0.2,
    subset           = "training",
    seed             = RANDOM_SEED,
    image_size       = IMAGE_SIZE,
    batch_size       = BATCH_SIZE,
    label_mode       = "int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    directory        = DATASET_DIR,
    validation_split = 0.2,
    subset           = "validation",
    seed             = RANDOM_SEED,
    image_size       = IMAGE_SIZE,
    batch_size       = BATCH_SIZE,
    label_mode       = "int"
)


# ==============================================================================
# CRITICAL: capture class_names BEFORE the pipeline transforms below.
# ==============================================================================
#
# image_dataset_from_directory() returns a DirectoryIteratorDataset which
# carries a .class_names attribute.  Once we apply .cache() / .shuffle() /
# .prefetch() the dataset is wrapped in a _PrefetchDataset which does NOT
# expose .class_names — accessing it would raise AttributeError.
#
# Solution: save the list into its own variable RIGHT HERE.
# ==============================================================================

CLASS_NAMES_DETECTED = train_ds.class_names   # e.g. ['Class_0_Clean', ...]


# ==============================================================================
# PERFORMANCE OPTIMISATION — prefetch
# ==============================================================================
#
# By default TensorFlow loads each batch just-in-time, meaning the GPU sits
# idle while the CPU reads the next batch from disk.
#
# .cache()    : After the first epoch, every image lives in RAM — no disk I/O.
# .prefetch() : While the GPU trains on batch N, the CPU prepares batch N+1.
#               AUTOTUNE lets TensorFlow decide the optimal buffer size.
# ==============================================================================

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


# ── Quick sanity check ────────────────────────────────────────────────────────
# NOTE: We use CLASS_NAMES_DETECTED (saved above), NOT train_ds.class_names,
# because train_ds is now a _PrefetchDataset and no longer has that attribute.
print(f"\nClass names (auto-detected): {CLASS_NAMES_DETECTED}")

# Peek at the shape of one batch: (batch_size, height, width, channels)
for images, labels in train_ds.take(1):
    print(f"Image batch shape : {images.shape}")
    print(f"Label batch shape : {labels.shape}")
    print(f"Pixel value range : [{images.numpy().min():.0f}, {images.numpy().max():.0f}]")
    print("  (values are 0-255 — the Rescaling layer inside the model will normalise them)")

## Cell 2 — CNN Model Architecture

In [ ]:
# ==============================================================================
# CELL 2 — MODEL ARCHITECTURE
# ==============================================================================
#
# We build a Sequential CNN — a linear stack of layers where the output of
# one layer is directly fed as the input to the next.
#
# THE BIG PICTURE: What does a CNN actually do?
# ──────────────────────────────────────────────
# A CNN learns to detect features (edges → textures → patterns → objects)
# in a hierarchical way:
#   - Early layers learn low-level features like horizontal/vertical edges.
#   - Middle layers combine those into shapes and textures.
#   - Deep layers recognise high-level concepts relevant to the class.
#
# For our DICE task:
#   - Clean images have sharp edges.
#   - Blurred images have smooth gradients (edges are gone).
#   - Noisy images have high-frequency random fluctuations everywhere.
# A CNN can learn to discriminate these patterns perfectly.
# ==============================================================================

model = keras.Sequential(name="DICE_CNN", layers=[

    # ── INPUT LAYER ────────────────────────────────────────────────────────────
    # Tells Keras the expected shape of one image: (height, width, channels).
    # channels = 3 for RGB. This layer does no computation.
    layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),

    # ── RESCALING LAYER ────────────────────────────────────────────────────────
    # Raw pixel values are integers in [0, 255].
    # Neural networks converge much faster with small floating-point inputs.
    # We divide by 255.0 to bring every value into [0.0, 1.0].
    # Doing this INSIDE the model means the saved model handles it automatically
    # at inference time — no risk of forgetting to pre-process at deployment.
    layers.Rescaling(1.0 / 255.0, name="normalise"),

    # ══════════════════════════════════════════════════════════════════════════
    # CONVOLUTIONAL BLOCK 1
    # ══════════════════════════════════════════════════════════════════════════
    #
    # ── WHY Conv2D? ───────────────────────────────────────────────────────────
    # Conv2D slides a small learnable filter (kernel) across the entire image.
    # At every position it computes a dot product between the filter weights
    # and the local pixel patch — this detects a specific feature at that spot.
    #
    # Parameters:
    #   filters  : Number of distinct feature detectors to learn.
    #              Block 1 → 32 filters (detects 32 different low-level features).
    #   kernel_size: The spatial size of each filter (3x3 is the modern standard;
    #              captures local context without being too expensive).
    #   activation: 'relu' (Rectified Linear Unit) sets all negative activations
    #              to 0.  This adds non-linearity, allowing the network to learn
    #              complex decision boundaries.  Without it, stacking layers is
    #              mathematically identical to a single linear transformation.
    #   padding  : 'same' pads the image with zeros so the output has the same
    #              spatial dimensions as the input (preserves feature maps).
    layers.Conv2D(filters=32, kernel_size=(3, 3), activation="relu",
                  padding="same", name="conv1a"),
    layers.Conv2D(filters=32, kernel_size=(3, 3), activation="relu",
                  padding="same", name="conv1b"),

    # ── WHY MaxPooling2D? ─────────────────────────────────────────────────────
    # After convolution we have a large feature map. MaxPooling2D divides it
    # into non-overlapping 2x2 windows and keeps only the MAXIMUM value in each.
    #
    # This achieves two goals simultaneously:
    #   1. DOWN-SAMPLING: Reduces the spatial size by half in each dimension
    #      (256x256 → 128x128), cutting computation for the next layer by 4x.
    #   2. TRANSLATION INVARIANCE: A feature detected slightly to the left or
    #      right of its 'ideal' position still produces the same max value —
    #      the network becomes robust to small positional shifts.
    layers.MaxPooling2D(pool_size=(2, 2), name="pool1"),
    # Output tensor shape after Block 1: (batch, 128, 128, 32)

    # ── DROPOUT ───────────────────────────────────────────────────────────────
    # During training, randomly sets 25% of activations to 0 each step.
    # This prevents co-adaptation (neurons relying on each other too much)
    # and is one of the most effective regularisation techniques.
    # Dropout is ONLY active during training — at inference it is a no-op.
    layers.Dropout(0.25, name="drop1"),

    # ══════════════════════════════════════════════════════════════════════════
    # CONVOLUTIONAL BLOCK 2
    # ══════════════════════════════════════════════════════════════════════════
    # We double the filter count to 64.
    # The spatial size shrinks (MaxPool halves it) so we can afford more filters
    # without exploding our memory budget.
    # These deeper filters combine the 32 simple features from Block 1 into
    # 64 higher-level composite features (e.g., textures, repeated patterns).
    layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu",
                  padding="same", name="conv2a"),
    layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu",
                  padding="same", name="conv2b"),
    layers.MaxPooling2D(pool_size=(2, 2), name="pool2"),
    # Output shape after Block 2: (batch, 64, 64, 64)
    layers.Dropout(0.25, name="drop2"),

    # ══════════════════════════════════════════════════════════════════════════
    # CONVOLUTIONAL BLOCK 3
    # ══════════════════════════════════════════════════════════════════════════
    # 128 filters — the deepest level of abstraction.
    # At this point, each filter responds to complex combinations of shapes
    # and textures across the entire receptive field.
    layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu",
                  padding="same", name="conv3a"),
    layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu",
                  padding="same", name="conv3b"),
    layers.MaxPooling2D(pool_size=(2, 2), name="pool3"),
    # Output shape after Block 3: (batch, 32, 32, 128)
    layers.Dropout(0.25, name="drop3"),

    # ══════════════════════════════════════════════════════════════════════════
    # CLASSIFIER HEAD
    # ══════════════════════════════════════════════════════════════════════════

    # ── FLATTEN ───────────────────────────────────────────────────────────────
    # The Conv layers produce 3-D tensors (height x width x filters).
    # Dense (fully-connected) layers require 1-D vectors.
    # Flatten unrolls (32, 32, 128) → a vector of 32x32x128 = 131,072 values.
    layers.Flatten(name="flatten"),

    # ── FULLY-CONNECTED (DENSE) LAYER ─────────────────────────────────────────
    # Every one of the 131,072 flattened values connects to every one of the
    # 256 neurons here. This lets the network combine global context from
    # anywhere in the image to make the final class decision.
    layers.Dense(256, activation="relu", name="dense1"),
    layers.Dropout(0.5, name="drop4"),   # Stronger dropout before the output

    # ── OUTPUT LAYER ──────────────────────────────────────────────────────────
    # 3 neurons — one per class (Clean / Blurred / Noisy).
    # NO activation function here: we output raw 'logits' (unnormalised scores).
    # SparseCategoricalCrossentropy(from_logits=True) in Cell 3 applies the
    # Softmax internally, which is numerically more stable than doing it
    # ourselves as a separate layer.
    # The class with the HIGHEST logit is the predicted class.
    layers.Dense(NUM_CLASSES, name="output_logits"),

])


# Print a human-readable table of every layer, its output shape, and param count
model.summary()

## Cell 3 — Compilation & Training

In [ ]:
# ==============================================================================
# CELL 3 — COMPILATION & TRAINING
# ==============================================================================
#
# 'Compiling' a Keras model means wiring the three components that drive
# every step of gradient-descent training:
#
#   1. OPTIMIZER  — decides how to update the weights given the gradient.
#   2. LOSS       — measures how wrong the current predictions are.
#   3. METRICS    — human-readable numbers to monitor progress (not used for
#                   gradient updates — purely for our visibility).
# ==============================================================================

model.compile(

    # ── Adam Optimizer ────────────────────────────────────────────────────────
    # 'adam' = Adaptive Moment Estimation.
    # It combines two techniques:
    #   - Momentum  — accumulates past gradients to dampen oscillations.
    #   - RMSProp   — divides the learning rate by the moving average of
    #                 recent gradient magnitudes, giving each parameter its
    #                 own adaptive learning rate.
    # Result: fast convergence with minimal hyperparameter tuning.
    # Default learning rate of 0.001 works well for most image tasks.
    optimizer = "adam",

    # ── Sparse Categorical Cross-Entropy Loss ─────────────────────────────────
    # 'Categorical' because we have multiple exclusive classes (not binary).
    # 'Sparse' because our labels are INTEGER indices (0, 1, 2), NOT one-hot
    # encoded vectors ([1,0,0], [0,1,0] ...). Sparse avoids that conversion step.
    # 'from_logits=True' tells the loss function that our model outputs raw
    # logits (no Softmax applied yet), so it applies Softmax internally for
    # numerically stable computation using the log-sum-exp trick.
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True),

    # ── Metrics ───────────────────────────────────────────────────────────────
    # 'accuracy' tracks what fraction of predictions are correct each epoch.
    # This is separate from the loss — accuracy is interpretable by humans,
    # while cross-entropy loss is what the optimizer actually minimises.
    metrics = ["accuracy"]

)


# ==============================================================================
# CALLBACKS — actions to run automatically at the end of each epoch
# ==============================================================================

callbacks = [

    # ── EarlyStopping ─────────────────────────────────────────────────────────
    # Monitors validation loss after every epoch.
    # If it hasn't improved for 3 consecutive epochs, training halts.
    # 'restore_best_weights=True' rewinds the model to the epoch where
    # validation loss was lowest — preventing the model from over-fitting.
    keras.callbacks.EarlyStopping(
        monitor              = "val_loss",
        patience             = 3,
        restore_best_weights = True,
        verbose              = 1
    ),

    # ── ReduceLROnPlateau ─────────────────────────────────────────────────────
    # If validation loss plateaus for 2 epochs, divide the learning rate by 5.
    # This lets the model take smaller steps when it's close to a minimum,
    # allowing finer weight adjustments for better accuracy.
    keras.callbacks.ReduceLROnPlateau(
        monitor   = "val_loss",
        factor    = 0.2,
        patience  = 2,
        min_lr    = 1e-6,
        verbose   = 1
    ),

]


# ==============================================================================
# TRAINING — model.fit()
# ==============================================================================
#
# What happens in ONE epoch:
#   For every batch in train_ds:
#     1. Forward pass  : images flow through all layers → predictions (logits)
#     2. Loss          : compare logits to true labels → scalar loss value
#     3. Backward pass : compute gradient of loss w.r.t. every weight
#     4. Optimizer step: update weights to reduce the loss
#   Then evaluate on val_ds (no weight updates, just measure generalisation).
#
# 'history' stores the per-epoch metrics — we'll plot them in Cell 4.
# ==============================================================================

print("Training DICE CNN...\n")
history = model.fit(
    train_ds,
    epochs          = EPOCHS,
    validation_data = val_ds,
    callbacks       = callbacks
)

## Cell 4 — Visualization

In [ ]:
# ==============================================================================
# CELL 4 — VISUALIZATION
# ==============================================================================
#
# Two key diagnostic plots:
#   A) Training vs. Validation Accuracy & Loss curves  — diagnose over/underfitting
#   B) Confusion Matrix                                 — per-class performance
# ==============================================================================


# ──────────────────────────────────────────────────────────────────────────────
# PART A — TRAINING HISTORY CURVES
# ──────────────────────────────────────────────────────────────────────────────
#
# history.history is a dict mapping metric names → list of per-epoch values.
# Keys: 'accuracy', 'val_accuracy', 'loss', 'val_loss'
#
# HOW TO READ THESE CURVES:
#   - Both curves rising together  → model is learning normally.
#   - Train acc high, val acc low  → OVERFITTING (model memorised training data).
#   - Both curves plateaued low    → UNDERFITTING (model too simple or lr wrong).
# ──────────────────────────────────────────────────────────────────────────────

acc      = history.history["accuracy"]
val_acc  = history.history["val_accuracy"]
loss     = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_ran = range(1, len(acc) + 1)

# Style settings for a clean, publication-ready look
plt.style.use("seaborn-v0_8-darkgrid")
COLORS = {
    "train" : "#4F8EF7",   # Blue for training
    "val"   : "#F76B4F",   # Orange-red for validation
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("DICE CNN — Training History",
             fontsize=18, fontweight="bold", y=1.02)

# ── Accuracy Plot ─────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_ran, acc,     color=COLORS["train"], lw=2.5,
        marker="o", markersize=6, label="Training Accuracy")
ax.plot(epochs_ran, val_acc, color=COLORS["val"],   lw=2.5,
        marker="s", markersize=6, linestyle="--", label="Validation Accuracy")
ax.set_title("Accuracy over Epochs", fontsize=14, fontweight="semibold")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_ylim([0, 1.05])
ax.legend(fontsize=11)
ax.annotate(f"Best Val: {max(val_acc):.3f}",
            xy=(val_acc.index(max(val_acc)) + 1, max(val_acc)),
            xytext=(10, -25), textcoords="offset points",
            fontsize=10, color=COLORS["val"],
            arrowprops=dict(arrowstyle="->", color=COLORS["val"]))

# ── Loss Plot ─────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs_ran, loss,     color=COLORS["train"], lw=2.5,
        marker="o", markersize=6, label="Training Loss")
ax.plot(epochs_ran, val_loss, color=COLORS["val"],   lw=2.5,
        marker="s", markersize=6, linestyle="--", label="Validation Loss")
ax.set_title("Loss over Epochs", fontsize=14, fontweight="semibold")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.legend(fontsize=11)
ax.annotate(f"Best Val: {min(val_loss):.3f}",
            xy=(val_loss.index(min(val_loss)) + 1, min(val_loss)),
            xytext=(10, 25), textcoords="offset points",
            fontsize=10, color=COLORS["val"],
            arrowprops=dict(arrowstyle="->", color=COLORS["val"]))

plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training history saved to training_history.png")


# ──────────────────────────────────────────────────────────────────────────────
# PART B — CONFUSION MATRIX
# ──────────────────────────────────────────────────────────────────────────────
#
# A confusion matrix is a square table where:
#   ROWS    = True class    (what the image actually is)
#   COLUMNS = Predicted class (what the model said it is)
#
# The DIAGONAL cells (top-left to bottom-right) are CORRECT predictions.
# OFF-DIAGONAL cells show where the model confused one class for another.
#
# Example: If row=1 (Blurred), col=0 (Clean) has value 12, it means
# 12 blurred images were wrongly classified as clean.
# ──────────────────────────────────────────────────────────────────────────────

# Step 1 — Run inference on the ENTIRE validation set
#   model.predict() returns raw logits for every image in val_ds.
#   We pick the index of the highest logit → predicted class integer.
print("\nGenerating predictions on the validation set...")
y_pred_logits = model.predict(val_ds, verbose=1)
y_pred        = np.argmax(y_pred_logits, axis=1)  # shape: (N,)

# Step 2 — Collect the true labels from the dataset
#   val_ds is a tf.data.Dataset of (images, labels) batches.
#   We concatenate all label tensors into one NumPy array.
y_true = np.concatenate([labels.numpy() for _, labels in val_ds])

# Step 3 — Build the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Step 4 — Normalise so each cell shows a PERCENTAGE of its true class
#   Dividing each row by its row-sum turns raw counts into proportions.
#   This is crucial when classes are imbalanced — raw counts can be misleading.
cm_normalised = cm.astype("float") / cm.sum(axis=1, keepdims=True)

# Short display labels (strip the 'Class_N_' prefix for cleaner axis ticks)
SHORT_NAMES = ["Clean", "Blurred", "Noisy"]

# Step 5 — Draw the heatmap
fig, ax = plt.subplots(figsize=(9, 7))

# Annotation strings: show both the normalised % and the raw count
annot_labels = np.array([
    [f"{cm_normalised[i,j]:.1%}\n({cm[i,j]})"
     for j in range(NUM_CLASSES)]
    for i in range(NUM_CLASSES)
])

sns.heatmap(
    cm_normalised,
    annot       = annot_labels,   # Custom annotation strings
    fmt         = "",             # 'fmt' must be empty when annot is a string array
    cmap        = "Blues",        # Light to Dark Blue gradient (high = blue)
    linewidths  = 0.8,            # Thin grid lines between cells
    linecolor   = "white",
    annot_kws   = {"size": 13, "weight": "bold"},
    xticklabels = SHORT_NAMES,
    yticklabels = SHORT_NAMES,
    vmin=0, vmax=1,              # Fix colour scale to [0, 1] for proportions
    ax=ax
)

# Build a colour bar label
cbar = ax.collections[0].colorbar
cbar.set_label("Proportion of True Class", fontsize=11)

ax.set_title("DICE CNN — Confusion Matrix (Validation Set)",
             fontsize=16, fontweight="bold", pad=18)
ax.set_xlabel("Predicted Label", fontsize=13, labelpad=10)
ax.set_ylabel("True Label",      fontsize=13, labelpad=10)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Confusion matrix saved to confusion_matrix.png")


# ──────────────────────────────────────────────────────────────────────────────
# PART C — CLASSIFICATION REPORT (bonus: precision / recall / F1 per class)
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  Classification Report")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=SHORT_NAMES))

## Cell 5 — Save the Model

In [ ]:
# ==============================================================================
# CELL 5 — SAVING THE MODEL
# ==============================================================================
#
# Keras supports two native save formats:
#
#   .keras   (recommended, used here)
#     - A single ZIP archive containing:
#         - model architecture (JSON)
#         - trained weights (HDF5 or shards)
#         - optimizer state (so training can be resumed)
#         - metadata (Keras version, etc.)
#     - Human-unreadable but fully self-contained and portable.
#
#   SavedModel (TensorFlow's lower-level format)
#     - A directory with a TensorFlow computation graph.
#     - Required for TensorFlow Serving / TFLite conversion.
#
# We use .keras because it is the modern Keras 3 default and is the easiest
# to reload later with keras.models.load_model().
# ==============================================================================

model.save(MODEL_PATH)   # Saves to ./dice_model.keras
print(f"Model saved to: {MODEL_PATH}")


# ── HOW TO RELOAD AND USE THE MODEL LATER ─────────────────────────────────────
#
#   loaded_model = keras.models.load_model('dice_model.keras')
#
#   # Prepare a single image for inference:
#   img = tf.keras.utils.load_img('path/to/image.jpg', target_size=(256, 256))
#   img_array = tf.keras.utils.img_to_array(img)   # shape: (256, 256, 3)
#   img_array = tf.expand_dims(img_array, axis=0)  # shape: (1, 256, 256, 3)
#
#   logits = loaded_model.predict(img_array)        # shape: (1, 3)
#   predicted_class = np.argmax(logits, axis=1)[0]  # 0, 1, or 2
#   print(CLASS_NAMES[predicted_class])
# ==============================================================================


# ── FINAL PROJECT SUMMARY ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  DICE PROJECT — COMPLETE SUMMARY")
print("=" * 60)
print(f"  Dataset      : VisDrone aerial drone imagery")
print(f"  Classes      : Clean | Blurred | Noisy")
print(f"  Architecture : Custom 3-block CNN")
print(f"  Epochs run   : {len(history.history['accuracy'])} / {EPOCHS}")
print(f"  Best Val Acc : {max(history.history['val_accuracy']):.4f}")
print(f"  Best Val Loss: {min(history.history['val_loss']):.4f}")
print(f"  Saved model  : {MODEL_PATH}")
print(f"  Saved plots  : training_history.png, confusion_matrix.png")
print("=" * 60)